# Session 10 · Activity 1 — RAGAS Evaluation with Cost Analysis

Evaluate the **real Lesson 10 RAG application** (`app/rag.py`) with two model backends over the
cat-health guide, scored by a single OpenAI judge, plus a manual token→cost comparison and
LangSmith tracing.

| | Chat model | Embedding model |
|---|---|---|
| **Arm A — Fireworks (open source)** | `gpt-oss-20b` | `qwen3-embedding-8b` (4096-dim) |
| **Arm B — OpenAI** | `gpt-4.1-mini` | `text-embedding-3-small` (1536-dim) |
| **RAGAS judge (both arms)** | OpenAI `gpt-5.4-mini` | — |

Both arms are built by importing and calling the **same** `app.rag.build_rag_graph` — we evaluate
the actual application, not a copy.

> ⚠️ **Educational only.** The cat-health content is course material, not veterinary advice.

**Why an isolated env + one pin:** RAGAS 0.4.3 hard-imports `langchain_community.chat_models.vertexai`,
which exists only in `langchain-community` 0.3.x. So this eval project pins
`langchain-community==0.3.31` (everything else is latest) and lives in its own `uv` env so the
app's normal environment (which needs community ≥0.4.1) is untouched.

**Cost strategy:** Fireworks credits are limited, so Fireworks is spent *only* on Arm A's RAG
generation + one-time corpus embedding. The judge and Arm B run on OpenAI (plentiful).


## Setup

Run once from `10_LLM_Servers/eval/`:

```bash
uv sync
uv run python -m ipykernel install --user --name aim-s10-eval --display-name "AIM S10 Eval"
```

Then select the **AIM S10 Eval** kernel for this notebook.

Required env vars (already present in `../.env`): `OPENAI_API_KEY`, `FIREWORKS_API_KEY`,
`FIREWORKS_CHAT_MODEL`, `FIREWORKS_EMBEDDING_MODEL`, `LANGSMITH_API_KEY`, `LANGSMITH_TRACING`,
`LANGSMITH_PROJECT`.


In [1]:
from __future__ import annotations

import asyncio
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Any

import instructor
import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from openai import OpenAI
from ragas.embeddings import embedding_factory
from ragas.llms import llm_factory
from ragas.metrics.collections import (
    AnswerAccuracy,
    AnswerCorrectness,
    ContextPrecisionWithReference,
    ContextRecall,
    Faithfulness,
)
from ragas.run_config import RunConfig

# Make the app package importable (repo layout: 10_LLM_Servers/eval/ -> parent has app/).
APP_ROOT = Path("..").resolve()
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

from app.rag import (  # noqa: E402  (import after sys.path tweak)
    FIREWORKS_BASE_URL,
    build_rag_graph,
    load_and_split,
)

/Users/msharp/projects/aiec1/10_LLM_Servers/eval/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the app's .env from the parent directory.
load_dotenv(APP_ROOT / ".env")

# --- Model configuration -------------------------------------------------
FIREWORKS_CHAT = os.environ.get("FIREWORKS_CHAT_MODEL", "accounts/fireworks/models/gpt-oss-20b")
FIREWORKS_EMBED = os.environ.get("FIREWORKS_EMBEDDING_MODEL", "accounts/fireworks/models/qwen3-embedding-8b")
OPENAI_CHAT = "gpt-4.1-mini"
OPENAI_EMBED = "text-embedding-3-small"
JUDGE_MODEL = "gpt-5.4-mini"          # OpenAI direct; NOT Fireworks
# Retrieval uses the real app's default retriever (k=4) so we evaluate the app as shipped.
# gpt-oss-20b is a reasoning model: it spends output tokens on hidden reasoning
# before the visible answer. A 512 cap gets fully consumed by reasoning and returns
# empty content, so give it enough headroom (verified ~700 tokens needed for a full answer).
GEN_MAX_TOKENS = 1024

# --- LangSmith tracing ---------------------------------------------------
os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "aim-session-10-llm-servers")

# --- RAGAS judge: synchronous OpenAI client with async-safe adapter ------
# (RAGAS metric methods call agenerate()/aembed_text(); a synchronous client raises
#  on those, and Instructor's AsyncOpenAI path is incompatible with the Jupyter event
#  loop. So keep the real requests synchronous and bridge only the coroutine boundary
#  with asyncio.to_thread.)
def build_sync_judge_llm():
    judge = llm_factory(
        JUDGE_MODEL,
        provider="openai",
        client=OpenAI(api_key=os.environ["OPENAI_API_KEY"]),  # default OpenAI base URL
    )
    # gpt-5.4-mini is a reasoning model and rejects `max_tokens` (needs
    # `max_completion_tokens`). RAGAS 0.4.3 auto-maps this ONLY for integer-versioned
    # names; it misparses the decimal "5.4" and would send max_tokens, causing a 400.
    # Setting model_args explicitly (no max_tokens key) sidesteps that.
    judge.model_args = {"max_completion_tokens": 2048, "max_retries": 3}

    async def agenerate_from_sync(prompt, response_model):
        return await asyncio.to_thread(judge.generate, prompt=prompt, response_model=response_model)

    judge.agenerate = agenerate_from_sync
    return judge


def build_judge_embeddings():
    # AnswerCorrectness needs an embedding model for its semantic-similarity component.
    # Use OpenAI (plentiful credits) so no Fireworks tokens are spent on judging.
    embeddings = embedding_factory(
        "openai",
        model=OPENAI_EMBED,
        client=OpenAI(api_key=os.environ["OPENAI_API_KEY"]),
    )

    # Bridge the async methods to the sync client (same reason as the judge above).
    async def aembed_text(text, **kwargs):
        return await asyncio.to_thread(embeddings.embed_text, text, **kwargs)

    async def aembed_texts(texts, **kwargs):
        return await asyncio.to_thread(embeddings.embed_texts, texts, **kwargs)

    embeddings.aembed_text = aembed_text
    embeddings.aembed_texts = aembed_texts
    return embeddings


judge_llm = build_sync_judge_llm()
if judge_llm.is_async:
    raise RuntimeError("RAGAS judge must be synchronous; reload the kernel and rerun this cell.")

ragas_run_config = RunConfig(timeout=180, max_retries=3, max_wait=30, max_workers=1)  # sequential


def run_ragas_sync(func, *args, **kwargs):
    with ThreadPoolExecutor(max_workers=1) as executor:
        return executor.submit(func, *args, **kwargs).result()


def run_ragas_async(async_function, *args, **kwargs):
    if asyncio.iscoroutine(async_function):
        return run_ragas_sync(asyncio.run, async_function)

    def invoke():
        return asyncio.run(async_function(*args, **kwargs))

    return run_ragas_sync(invoke)


print("Arm A (Fireworks):", FIREWORKS_CHAT, "| embed", FIREWORKS_EMBED)
print("Arm B (OpenAI):    ", OPENAI_CHAT, "| embed", OPENAI_EMBED)
print("Judge:             ", JUDGE_MODEL, "(OpenAI)")
print("LangSmith project: ", os.environ["LANGSMITH_PROJECT"], "| tracing", os.environ["LANGSMITH_TRACING"])

Arm A (Fireworks): accounts/fireworks/models/gpt-oss-20b | embed accounts/fireworks/models/qwen3-embedding-8b
Arm B (OpenAI):     gpt-4.1-mini | embed text-embedding-3-small
Judge:              gpt-5.4-mini (OpenAI)
LangSmith project:  aim-session-10-llm-servers | tracing true


## Build both arms from the real app

We split the PDF **once**, then call the real `app.rag.build_rag_graph` twice with different
models. The two arms need **separate Qdrant collections** because their embedding dimensions
differ (Fireworks 4096-dim vs OpenAI 1536-dim).


In [3]:
# Split the corpus ONCE (both arms embed the same chunks into their own collections).
chunks = load_and_split(str(APP_ROOT / "data"))
print(f"Loaded {len(chunks)} chunks from the cat-health PDF.")

# Arm A — Fireworks (the ONLY place Fireworks credits are spent).
armA_chat = ChatOpenAI(
    model=FIREWORKS_CHAT,
    openai_api_key=os.environ["FIREWORKS_API_KEY"],
    openai_api_base=FIREWORKS_BASE_URL,
    max_tokens=GEN_MAX_TOKENS,
    temperature=0,
)
armA_embed = OpenAIEmbeddings(
    model=FIREWORKS_EMBED,
    openai_api_key=os.environ["FIREWORKS_API_KEY"],
    openai_api_base=FIREWORKS_BASE_URL,
    check_embedding_ctx_length=False,
    dimensions=4096,
)
armA_graph = build_rag_graph(
    chat_model=armA_chat, embedding_model=armA_embed,
    collection_name="cat_health_fireworks", chunks=chunks,
)

# Arm B — OpenAI.
armB_chat = ChatOpenAI(model=OPENAI_CHAT, api_key=os.environ["OPENAI_API_KEY"],
                       max_tokens=GEN_MAX_TOKENS, temperature=0)
armB_embed = OpenAIEmbeddings(model=OPENAI_EMBED, api_key=os.environ["OPENAI_API_KEY"])
armB_graph = build_rag_graph(
    chat_model=armB_chat, embedding_model=armB_embed,
    collection_name="cat_health_openai", chunks=chunks,
)
print("Both arms built from app.rag.build_rag_graph.")

Loaded 42 chunks from the cat-health PDF.
Both arms built from app.rag.build_rag_graph.


## Golden dataset

A small hand-authored set of questions with reference answers drawn directly from the source PDF
— the **2021 AAHA/AAFP Feline Life Stage Guidelines**. The questions span life stages,
vaccination, weight/nutrition, parasite control, behavior/pain, exam frequency, and elimination.
Keeping the set tiny conserves Fireworks credits and avoids rate limits.

Each `reference` is grounded in the document, so the answer-vs-reference metrics
(`answer_accuracy`, `answer_correctness`) and `context_recall` are meaningful.

In [11]:
# Golden Q&A grounded in the source PDF: the 2021 AAHA/AAFP Feline Life Stage
# Guidelines. Every reference below is taken directly from that document, spanning
# life stages, vaccination, weight/nutrition, parasite control, behavior/pain, and
# general care / elimination.
GOLDEN = [
    {"user_input": "What are the four age-related feline life stages defined in the guidelines, and what age ranges do they cover?",
     "reference": "The four age-related life stages are kitten (birth up to 1 year), young adult (1 through 6 years), mature adult (7 to 10 years), and senior (over 10 years). A fifth stage, end of life, can occur at any age and is not age-specific."},
    {"user_input": "Which vaccines are considered core vaccines for cats?",
     "reference": "The core vaccines are rabies virus, feline herpesvirus type 1 (FHV-1), feline calicivirus (FCV), and feline panleukopenia virus (FPV). Feline leukemia virus (FeLV) vaccination is also considered core for kittens and young cats."},
    {"user_input": "How is a cat classified as overweight or obese using body condition score, and why does excess weight matter?",
     "reference": "A body condition score (BCS) of 6/9 or 7/9 is considered overweight and a score of 8/9 or greater is considered obese. Excess weight matters because being overweight or obese can predispose cats to chronic conditions such as diabetes mellitus, lameness (related to osteoarthritis and soft-tissue injury), non-allergic skin disease, urethral obstruction, and oral disease."},
    {"user_input": "What parasite treatment is recommended for kittens and newly adopted cats with an unknown history of medical care?",
     "reference": "For kittens and newly adopted cats with an unknown history, it is prudent to give prophylactic treatment with broad-spectrum products effective against heartworms, intestinal parasites, and fleas. Canine and feline housemates should be treated at the same time because they may be at risk of transmission of parasites such as roundworm and fleas."},
    {"user_input": "What behavioral changes might indicate degenerative joint disease (DJD) or pain in a senior cat?",
     "reference": "DJD or muscle weakness in senior cats often first shows up as reduced or changed jumping and climbing, for example owners reporting the cat is 'not getting on the counters as much' or 'doesn't like his window seat anymore.' Reduced grooming can also indicate DJD pain, bladder pain, or reduced mobility."},
    {"user_input": "How often does the Task Force recommend that cats receive veterinary examinations?",
     "reference": "The Task Force recommends a minimum of annual examinations for all cats, with increasing frequency as appropriate for the individual. Senior cats should be seen at least every 6 months, and more frequently if they have chronic conditions."},
    {"user_input": "What is the rule of thumb for how many litter boxes a household should provide?",
     "reference": "The rule of thumb is one litter box for each cat plus one additional box, or one litter box for each social group plus one additional box if the number of social groups is known. Boxes should be placed in multiple quiet, easily accessible locations, particularly in multicat households."},
]
golden_df = pd.DataFrame(GOLDEN)
golden_df

,user_input,reference
0,What are the four age-related feline life stag...,The four age-related life stages are kitten (b...
1,Which vaccines are considered core vaccines fo...,"The core vaccines are rabies virus, feline her..."
2,How is a cat classified as overweight or obese...,A body condition score (BCS) of 6/9 or 7/9 is ...
3,What parasite treatment is recommended for kit...,For kittens and newly adopted cats with an unk...
4,What behavioral changes might indicate degener...,DJD or muscle weakness in senior cats often fi...
5,How often does the Task Force recommend that c...,The Task Force recommends a minimum of annual ...
6,What is the rule of thumb for how many litter ...,The rule of thumb is one litter box for each c...


## Run each arm, capturing token usage

`run_arm` invokes the real graph per question and reads token usage from the raw `AIMessage`
(preserved by the refactored `generate` node). This is one generation call per query — no
double Fireworks spend.


In [12]:
def run_arm(graph, df: pd.DataFrame, arm_name: str) -> list[dict[str, Any]]:
    rows = []
    for _, r in df.iterrows():
        out = graph.invoke({"question": r["user_input"]})
        msg = out.get("response_message")
        usage = getattr(msg, "usage_metadata", None) or {}
        # Fallback for providers that only populate response_metadata.
        if not usage and msg is not None:
            usage = (msg.response_metadata or {}).get("token_usage", {}) or {}
        rows.append({
            "arm": arm_name,
            "user_input": r["user_input"],
            "reference": r["reference"],
            "retrieved_contexts": [d.page_content for d in out.get("context", [])],
            "response": out["response"],
            "input_tokens": usage.get("input_tokens", usage.get("prompt_tokens", 0)),
            "output_tokens": usage.get("output_tokens", usage.get("completion_tokens", 0)),
        })
    return rows

In [13]:
armA_rows = run_arm(armA_graph, golden_df, "fireworks_gpt_oss_20b")
armB_rows = run_arm(armB_graph, golden_df, "openai_gpt_4_1_mini")
pd.DataFrame(armA_rows + armB_rows)[["arm", "user_input", "response", "input_tokens", "output_tokens"]]

,arm,user_input,response,input_tokens,output_tokens
0,fireworks_gpt_oss_20b,What are the four age-related feline life stag...,The guidelines define four age‑related feline ...,3211,241
1,fireworks_gpt_oss_20b,Which vaccines are considered core vaccines fo...,The core vaccines recommended for cats are:\n\...,3974,283
2,fireworks_gpt_oss_20b,How is a cat classified as overweight or obese...,A cat is considered **overweight** when its bo...,3957,432
3,fireworks_gpt_oss_20b,What parasite treatment is recommended for kit...,For kittens and newly adopted cats with an unk...,4146,155
4,fireworks_gpt_oss_20b,What behavioral changes might indicate degener...,Behavioral clues that a senior cat may be expe...,3934,720
5,fireworks_gpt_oss_20b,How often does the Task Force recommend that c...,The Task Force recommends a minimum of **annua...,3164,151
6,fireworks_gpt_oss_20b,What is the rule of thumb for how many litter ...,The rule of thumb is to provide **one litter b...,4143,139
7,openai_gpt_4_1_mini,What are the four age-related feline life stag...,The four age-related feline life stages define...,2875,63
8,openai_gpt_4_1_mini,Which vaccines are considered core vaccines fo...,"The core vaccines for cats are rabies virus, f...",3913,68
9,openai_gpt_4_1_mini,How is a cat classified as overweight or obese...,A cat is classified as overweight if it has a ...,3619,104


## Score both arms with the single OpenAI judge

The same `gpt-5.4-mini` judge scores every row of both arms. Metrics map to the assignment's
three requirements, plus two extras for depth:

- **Faithfulness** → answer faithfulness (grounded in retrieved context)
- **ContextRecall** → retrieval quality (did we retrieve what the reference needs?)
- **AnswerAccuracy** → end-to-end accuracy vs the reference
- **ContextPrecisionWithReference** + **AnswerCorrectness** → extra retrieval/answer depth


In [14]:
async def score_rows(rows: list[dict[str, Any]]) -> pd.DataFrame:
    j = build_sync_judge_llm()      # fresh judge per call (Lesson 06 pattern)
    e = build_judge_embeddings()    # OpenAI embeddings for AnswerCorrectness similarity
    m = {
        "faithfulness": Faithfulness(llm=j),
        "context_recall": ContextRecall(llm=j),
        "context_precision": ContextPrecisionWithReference(llm=j),
        "answer_accuracy": AnswerAccuracy(llm=j),
        "answer_correctness": AnswerCorrectness(llm=j, embeddings=e),
    }
    scored = []
    for i, r in enumerate(rows, start=1):
        scored.append({
            "arm": r["arm"],
            "case": i,
            "faithfulness": (await m["faithfulness"].ascore(
                user_input=r["user_input"], response=r["response"],
                retrieved_contexts=r["retrieved_contexts"])).value,
            "context_recall": (await m["context_recall"].ascore(
                user_input=r["user_input"], retrieved_contexts=r["retrieved_contexts"],
                reference=r["reference"])).value,
            "context_precision": (await m["context_precision"].ascore(
                user_input=r["user_input"], reference=r["reference"],
                retrieved_contexts=r["retrieved_contexts"])).value,
            "answer_accuracy": (await m["answer_accuracy"].ascore(
                user_input=r["user_input"], response=r["response"],
                reference=r["reference"])).value,
            "answer_correctness": (await m["answer_correctness"].ascore(
                user_input=r["user_input"], response=r["response"],
                reference=r["reference"])).value,
        })
    return pd.DataFrame(scored)

In [15]:
armA_scores = run_ragas_async(score_rows, armA_rows)
armB_scores = run_ragas_async(score_rows, armB_rows)
pd.concat([armA_scores, armB_scores], ignore_index=True)

/var/folders/r6/1fwxrpyn1w937l1v73wpn3jc0000gn/T/ipykernel_48130/3189578504.py:47: DeprecationWarning: Importing embedding_factory from ragas.embeddings is deprecated. Import directly from ragas.embeddings.base or use modern providers: from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embeddings = embedding_factory(


,arm,case,faithfulness,context_recall,context_precision,answer_accuracy,answer_correctness
0,fireworks_gpt_oss_20b,1,0.666667,1.0,0.916667,0.50,0.600864
1,fireworks_gpt_oss_20b,2,1.000000,1.0,0.833333,1.00,0.914822
2,fireworks_gpt_oss_20b,3,1.000000,1.0,0.583333,1.00,0.931404
3,fireworks_gpt_oss_20b,4,1.000000,1.0,0.833333,0.50,0.817676
4,fireworks_gpt_oss_20b,5,0.851852,1.0,1.000000,0.50,0.810300
5,fireworks_gpt_oss_20b,6,1.000000,1.0,1.000000,1.00,0.879506
6,fireworks_gpt_oss_20b,7,1.000000,1.0,1.000000,0.75,0.731685
7,openai_gpt_4_1_mini,1,1.000000,1.0,0.916667,1.00,0.786649
8,openai_gpt_4_1_mini,2,1.000000,1.0,1.000000,0.75,0.992588
9,openai_gpt_4_1_mini,3,1.000000,1.0,1.000000,1.00,0.988908


## Cost analysis (manual token→cost table)

LangSmith auto-prices OpenAI but **not** Fireworks custom endpoints, so we compute cost manually
from captured token counts. Query-embedding tokens are negligible; corpus embedding is a one-time
Fireworks cost. To project "at scale," multiply `avg_cost_per_query_usd` by expected query volume.

> 💲 **Verify `PRICES`** against current provider pricing pages before quoting numbers in the Loom.


In [16]:
# USD per 1,000,000 tokens. TODO: VERIFY against current provider pricing pages.
# Defaults below are placeholders — confirm before quoting numbers in the Loom.
PRICES = {
    FIREWORKS_CHAT:  {"input": 0.10, "output": 0.40},   # TODO verify (Fireworks gpt-oss-20b)
    OPENAI_CHAT:     {"input": 0.40, "output": 1.60},    # TODO verify (OpenAI gpt-4.1-mini)
}


def query_cost(model: str, in_tok: int, out_tok: int) -> float:
    p = PRICES[model]
    return in_tok / 1e6 * p["input"] + out_tok / 1e6 * p["output"]


cost_rows = []
for arm_name, rows, model in [
    ("fireworks_gpt_oss_20b", armA_rows, FIREWORKS_CHAT),
    ("openai_gpt_4_1_mini", armB_rows, OPENAI_CHAT),
]:
    for r in rows:
        cost_rows.append({
            "arm": arm_name,
            "input_tokens": r["input_tokens"],
            "output_tokens": r["output_tokens"],
            "cost_usd": query_cost(model, r["input_tokens"], r["output_tokens"]),
        })
cost_df = pd.DataFrame(cost_rows)
cost_summary = cost_df.groupby("arm").agg(
    avg_input_tokens=("input_tokens", "mean"),
    avg_output_tokens=("output_tokens", "mean"),
    avg_cost_per_query_usd=("cost_usd", "mean"),
    total_cost_usd=("cost_usd", "sum"),
)
cost_summary

,avg_input_tokens,avg_output_tokens,avg_cost_per_query_usd,total_cost_usd
arm,,,,
fireworks_gpt_oss_20b,3789.857143,303.000000,0.000500,0.003501
openai_gpt_4_1_mini,3728.857143,75.142857,0.001612,0.011282


## Quality comparison summary

In [17]:
quality = pd.concat([armA_scores, armB_scores], ignore_index=True)
quality.groupby("arm").mean(numeric_only=True).drop(columns=["case"]).T

arm,fireworks_gpt_oss_20b,openai_gpt_4_1_mini
faithfulness,0.931217,1.000000
context_recall,1.000000,1.000000
context_precision,0.880952,0.936508
answer_accuracy,0.750000,0.750000
answer_correctness,0.812323,0.784069


## Interpretation & takeaways

Use this section (and the Loom) to discuss:

- **Quality vs cost tradeoff.** Compare the open-source `gpt-oss-20b` arm against `gpt-4.1-mini`
  across faithfulness, retrieval quality, and end-to-end accuracy, alongside cost per query.
- **Fair comparison.** A single fixed OpenAI judge (`gpt-5.4-mini`) scored both arms, so
  differences reflect the RAG backends, not judge drift.
- **Traces.** Open the LangSmith dashboard for project `aim-session-10-llm-servers` to show the
  per-run retrieve → generate spans for both arms.
- **At scale.** Multiply `avg_cost_per_query_usd` by your expected monthly query volume to project
  the total cost of each provider.
